# AE and ELM labels of one shot, with their data

Type a shot number and run all. For both all-shot databases in `/projects/EKOLEMEN/tokamaksi/diiid/release`
it shows the 1 ms label (shaded), the model probability it was thresholded from, and the input data:

- **AE**: the CO2 spectrogram (80-250 kHz, the model's own input, chords averaged). The CO2 records were
  deleted after labeling, so it is shown only for the 4,875 tier shots and for shots fetched with the
  command below.
- **ELM**: the filterscopes (FS02-04, the maximum per 0.1 ms) and the CO2 density chords DENV2F/DENV3F,
  kept for every labeled shot.

AE spectrogram of any other shot (login node, needs a live `fdp login`; ~10 s and ~15 MB a shot):

    cd /scratch/gpfs/EKOLEMEN/nc1514/FusionAIHub
    pixi run --frozen -e fdp fdp -D d3d run python scripts/labeler/ae_fdp_label.py view 125000 141211

Kernel: any pixi env with pandas, pyarrow and matplotlib (`labeler`).

In [ ]:
SHOT = 156796
# SHOT = random_shot()           # a random shot with both labels; random_shot(era=125000) for one era

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

LABELS = Path("/projects/EKOLEMEN/tokamaksi/diiid/release")
CACHE = Path("/scratch/gpfs/EKOLEMEN/nc1514/FusionAIHub/.cache/labeler")
DB = {"AE": LABELS / "alfven_eigenmode/ae_seldnet_fdp_1ms", "ELM": LABELS / "edge_localized_mode/elm_fdp_1ms"}
PROB = {"AE": CACHE / "ae_fdp_1ms/work", "ELM": CACHE / "elm_fdp_1ms/work"}      # model probabilities
ELM_INPUTS = CACHE / "elm_fdp_1ms/inputs"                                         # every labeled shot
AE_SPECS = [CACHE / "ae_tiers/specs", CACHE / "ae_view"]                          # tier shots; fetched shots
shots = {k: pd.read_parquet(d / "shots.parquet").set_index("shot") for k, d in DB.items()}
grids = {k: pd.read_csv(d / "grid_1ms/index.csv").set_index("shot") for k, d in DB.items()}
INK, MUTED, LABEL = "#0b0b0b", "#52514e", "#2a78d6"


def label_1ms(event, shot):
    """(t_ms, 0/1 label) on the assessed window, or None with the shot's status."""
    g = grids[event]
    if shot not in g.index:
        return None, shots[event].status.get(shot, "not a plasma shot")
    n = int(g.end_ms[shot] - g.start_ms[shot])
    lab = np.unpackbits(np.fromfile(DB[event] / f"grid_1ms/{shot // 1000}/{shot}.bin", np.uint8), count=n)
    return (g.start_ms[shot] + np.arange(n), lab), "labeled"


def random_shot(era=None, seed=None):
    both = grids["AE"].index.intersection(grids["ELM"].index)
    if era is not None:
        both = both[(both >= era) & (both < era + 5000)]
    return int(np.random.default_rng(seed).choice(both))


def shade(ax, lab):
    t, y = lab
    ax.fill_between(t, 0, 1, where=y.astype(bool), step="post", color=LABEL, alpha=0.25, lw=0,
                    transform=ax.get_xaxis_transform())


def show(shot, xlim=None):
    ae, ae_status = label_1ms("AE", shot)
    elm, elm_status = label_1ms("ELM", shot)
    spec = next((d / f"{shot}.npy" for d in AE_SPECS if (d / f"{shot}.meta.npz").exists()), None)
    ae_p, elm_p, elm_in = (PROB["AE"] / f"{shot}.npz", PROB["ELM"] / f"{shot}.npz", ELM_INPUTS / f"{shot}.npz")
    panels = [name for name, ok in (("spec", spec is not None), ("ae", ae_p.exists()), ("fs", elm_in.exists()),
                                    ("dens", elm_in.exists()), ("elm", elm_p.exists())) if ok]
    if not panels:
        print(f"{shot}: AE {ae_status}, ELM {elm_status}; nothing to plot")
        return
    heights = {"spec": 2.2, "ae": 1, "fs": 1.4, "dens": 1, "elm": 1}
    fig, axes = plt.subplots(len(panels), 1, sharex=True, squeeze=False, figsize=(12, 0.6 + 1.3 * sum(heights[p] for p in panels)),
                             height_ratios=[heights[p] for p in panels])
    ax = dict(zip(panels, axes[:, 0]))
    if "spec" in ax:
        s = np.load(spec, mmap_mode="r")
        s = (s.mean(axis=1, dtype=np.float32) if s.ndim == 3 else s.astype(np.float32)).T  # (bins, columns)
        ts = np.load(spec.with_suffix(".meta.npz"))["t_ms"]
        ax["spec"].imshow(s, origin="lower", aspect="auto", cmap="magma", extent=(ts[0], ts[-1], 80, 250),
                          vmin=np.percentile(s, 5), vmax=np.percentile(s, 99.5), interpolation="none")
        ax["spec"].set_ylabel("CO2 kHz")
    for event, key, path in (("AE", "ae", ae_p), ("ELM", "elm", elm_p)):
        if key not in ax:
            continue
        w = np.load(path)
        n = w["prob"].size
        t = w["t_first_ms"] + w["hop_ms"] * np.arange(n)
        thr = float(w["threshold"]) if "threshold" in w.files else 0.5
        lab = ae if event == "AE" else elm
        if lab is not None:
            shade(ax[key], lab)
        ax[key].plot(t, w["prob"].astype(np.float32), lw=0.6, c=INK)
        ax[key].axhline(thr, c=MUTED, lw=0.8, ls="--")
        ax[key].set(ylim=(0, 1), ylabel=f"{event} p")
    if "fs" in ax:
        z = np.load(elm_in)
        t = float(z["grid0_ms"]) + float(z["dt_ms"]) * np.arange(z["fmax"].shape[1])
        fs = np.nanmax(z["fmax"], axis=0)
        ax["fs"].plot(t, fs, lw=0.4, c=INK)
        ax["fs"].set(yscale="log", ylabel="FS02-04 max")
        lo, hi = np.nanpercentile(fs[fs > 0], [1, 99.95]) if np.any(fs > 0) else (1, 10)
        ax["fs"].set_ylim(lo / 2, hi * 2)
        if elm is not None:
            shade(ax["fs"], elm)
        for row, (name, c) in enumerate(zip(("DENV2F", "DENV3F"), ("#2a78d6", "#eb6834"))):
            ax["dens"].plot(t, z["dmean"][row], lw=0.4, c=c, label=name)
        ax["dens"].set(ylabel="CO2 density")
        for line in ax["dens"].legend(loc="upper left", fontsize=8, frameon=False, ncols=2).get_lines():
            line.set_linewidth(2)
    end = max(np.nan_to_num(float(shots[k].plasma_end_ms.get(shot, 0))) for k in shots)
    axes[-1, 0].set(xlabel="time (ms)", xlim=xlim or (-50, end + 100))
    def summary(event, lab, status):
        return f"{event} {lab[1].sum():,} of {lab[1].size:,} ms" if lab is not None else f"{event} {status}"
    axes[0, 0].set_title(f"{shot}: {summary('AE', ae, ae_status)}; {summary('ELM', elm, elm_status)} "
                         f"(blue: labeled present; dashed: threshold)", fontsize=10, loc="left")
    fig.align_ylabels()
    plt.show()

In [ ]:
show(SHOT)  # show(SHOT, xlim=(1500, 2500)) to zoom